# Lab 76: The Edge of Sleep

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 9, *The Future Interface*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-76.ipynb)

**What you will do:** try the Dalí spoon-and-plate method (or the smartphone voice-recorder variant) to catch your own hypnagogic imagery on the way to sleep, log what you noticed, and then watch a small diffusion model run the same noise-to-structure process AI researchers call "denoising".

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 76 you lay down in a dim room and tried to catch the **hypnagogic state** -- the few minutes
between wakefulness and sleep -- using either the spoon-and-plate method or a quiet voice recorder. As
the thalamus gated sensory input, your brain's generative model kept running, producing spontaneous
imagery (faces, geometric patterns, scenes) with nothing external driving it.

## Record your results

You cannot quantify a face or a spiral, but you can quantify *when* things happened and *what kind* of
imagery you saw. Only try this when it is actually safe for you to doze off for a few minutes (not
before driving, and not when you need to stay alert). Log one row per session below -- or per night, if
you tried the Dalí method over a week -- then run the cell. This is a personal log, not a controlled
experiment: the point is to notice patterns across your own attempts, not to prove anything.

In [ ]:
import pandas as pd

# example data: replace with your own sessions
sessions = pd.DataFrame([
    {"session": 1, "minutes_to_first_image": 6, "content": "geometric pattern", "vivid": 3},
    {"session": 2, "minutes_to_first_image": 4, "content": "a stranger's face",  "vivid": 4},
    {"session": 3, "minutes_to_first_image": 9, "content": "fragment of speech", "vivid": 2},
    {"session": 4, "minutes_to_first_image": 5, "content": "falling sensation",  "vivid": 4},
])  # vivid: 1 (faint) to 5 (as real as being awake)

display(sessions)
sessions["content"].value_counts().plot.barh(color="steelblue", figsize=(6, 2.5))
plt.xlabel("number of sessions"); plt.title("What you saw, across sessions"); plt.gca().invert_yaxis()
plt.show()
print(f"Average time to first image: {sessions['minutes_to_first_image'].mean():.1f} minutes")

## The AI side

A **diffusion model** generates images the way your brain generated hypnagogic imagery: it starts from
pure noise and progressively imposes structure, guided only by patterns it learned earlier, with no
external image telling it what to draw. Below, a small diffusion model trained on tiny 32x32 photographs
(Ho, Jain, & Abbeel, 2020) denoises pure static into a picture, one step at a time. Watch the early
steps -- blobs and colour fields, not yet anything -- and compare them with the geometric, not-quite-
formed patterns you may have logged above.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q diffusers

import torch
from diffusers import DDPMPipeline, DDIMScheduler

pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32")
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
n_steps = 30
pipe.scheduler.set_timesteps(n_steps)

generator = torch.manual_seed(2026)
image = torch.randn((1, 3, 32, 32), generator=generator)
snapshot_at = {0, 5, 10, 15, 20, 25, n_steps - 1}
snapshots, labels = [], []

with torch.no_grad():
    for i, t in enumerate(pipe.scheduler.timesteps):
        noise_pred = pipe.unet(image, t).sample
        image = pipe.scheduler.step(noise_pred, t, image).prev_sample
        if i in snapshot_at:
            frame = (image / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
            snapshots.append(frame)
            labels.append(f"step {i + 1}/{n_steps}")

fig, axes = plt.subplots(1, len(snapshots), figsize=(2.1 * len(snapshots), 2.3))
for ax, frame, label in zip(axes, snapshots, labels):
    ax.imshow(frame)
    ax.set_title(label, fontsize=9)
    ax.axis("off")
plt.suptitle("A diffusion model 'dreaming': noise becoming structure")
plt.tight_layout()
plt.show()

Notice how the earliest frames are abstract colour and texture -- not yet a face or an object -- and
how structure crystallises only in the later steps, just as your own hypnagogic images tend to sharpen
for a moment before you wake yourself to record them. The analogy has real limits: this model was
trained on millions of labelled photographs and is denoising towards *some* photograph, whereas your
brain's hypnagogic imagery is shaped by a lifetime of embodied experience and is not aiming at any
particular target. The model also has no thalamus, no sleep stages, and nothing it is like to be it
while it denoises -- it resembles the *process* of hypnagogia, not a mind that has hypnagogic
experiences.

## Questions to think about

1. Look at your log. Did geometric patterns tend to appear earlier in a session than faces or scenes, similar to how the diffusion model's early steps were abstract and its later steps more object-like?
2. The book says the hypnagogic state is what happens when the precision of sensory prediction errors is "turned down". What would happen to the diffusion model's output if you fed it a very noisy real photograph instead of pure random noise -- would that correspond to higher or lower sensory precision?
3. The AI Connection notes that "familiar patterns (especially faces) tend to emerge early" in both systems. Did any of your own recorded sessions include a face? If several people tried this Lab, would you expect faces to be the single most common content, and why might that be true of a face-detecting prediction machine specifically?
4. Diffusion models are trained to denoise towards a specific target that the training data defines. Your brain has no equivalent externally imposed target during hypnagogia. What does this difference suggest about how much weight to put on the analogy?

## Challenge

Try the Dalí method (or the smartphone voice-recorder variant) on three separate nights, this time noting
not just the content of each image but how quickly it decayed once you tried to hold onto it after waking
yourself. Then look back at the diffusion model's snapshots above: does the speed of your own image's
decay resemble anything about how quickly the model's structure would unravel if the denoising steps ran
backwards (adding noise back in)?

In [ ]:
# Example data: replace with your own three-night log.
# decay_seconds = how long the image stayed vivid enough to describe, after you woke yourself
nights = pd.DataFrame([
    {"night": 1, "content": "spiral pattern", "decay_seconds": 8},  # example data
    {"night": 2, "content": "a face",         "decay_seconds": 4},  # example data
    {"night": 3, "content": "a voice",        "decay_seconds": 6},  # example data
])
display(nights)
print(f"Average decay time: {nights['decay_seconds'].mean():.1f} seconds")
print("This is far too small an n to draw a real conclusion -- it is a way of noticing your own pattern,")
print("not a test of a general claim about hypnagogic decay rates.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-76.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")